In [1]:
import torch

print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip install -q transformers accelerate sentencepiece datasets evaluate textstat sentence-transformers sacrebleu psutil huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.8 MB/s eta 0:00:00


In [5]:
import torch
import pandas as pd
import json
import time
import psutil

from transformers import AutoTokenizer, AutoModelForCausalLM

In [6]:
from google.colab import files

uploaded = files.upload()

Saving Medical_Simplification_Translation.ipynb to Medical_Simplification_Translation.ipynb


In [7]:
import json

medical_data = [
    {
        "medical": "The patient is suffering from hypertension."
    },
    {
        "medical": "The patient has diabetes mellitus."
    },
    {
        "medical": "The patient is diagnosed with pneumonia."
    },
    {
        "medical": "The patient is experiencing chest pain."
    },
    {
        "medical": "The patient has chronic kidney disease."
    }
]

with open("medical.json", "w") as f:
    json.dump(medical_data, f, indent=4)

print("medical.json created successfully!")

medical.json created successfully!


In [8]:
import json

with open("medical.json", "r") as f:
    data = json.load(f)

print(data)

[{'medical': 'The patient is suffering from hypertension.'}, {'medical': 'The patient has diabetes mellitus.'}, {'medical': 'The patient is diagnosed with pneumonia.'}, {'medical': 'The patient is experiencing chest pain.'}, {'medical': 'The patient has chronic kidney disease.'}]


In [9]:
from huggingface_hub import login

login()

In [10]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "meta-llama/Llama-3.2-1B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
start = time.time()

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

end = time.time()

print("✅ Model Loaded Successfully!")
print(f"Load Time: {end-start:.2f} seconds")

Loading tokenizer...


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model Loaded Successfully!
Load Time: 21.61 seconds


In [11]:
medical_sentence = "The patient is suffering from hypertension."

prompt = f"""
Simplify the following medical sentence into simple English.

Medical Sentence:
{medical_sentence}

Simple English:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.3,
    do_sample=False
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Simplify the following medical sentence into simple English.

Medical Sentence:
The patient is suffering from hypertension.

Simple English:
The patient is having high blood pressure.

Note: This is a common medical term that refers to a condition where the blood pressure in the body is too high. It is a serious condition that requires medical attention.


In [13]:
simplified_sentences = []

for item in data:
    medical_sentence = item["medical"]

    prompt = f"""Simplify this medical sentence into simple English.

Medical: {medical_sentence}

Simple:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Keep only the generated answer
    simple_text = generated_text.replace(prompt, "").strip()

    simplified_sentences.append({
        "medical": medical_sentence,
        "simplified": simple_text
    })

for item in simplified_sentences:
    print("Medical :", item["medical"])
    print("Simple  :", item["simplified"])
    print("-" * 60)

Medical : The patient is suffering from hypertension.
Simple  : The patient has high blood pressure.

Note: This is a common medical term that is often used to describe a condition where the blood pressure in the body is too high.
------------------------------------------------------------
Medical : The patient has diabetes mellitus.
Simple  : The patient has diabetes.

Explanation:

*   **Diabetes mellitus**: A medical term that describes a group of diseases that affect the way the body processes blood sugar (glucose). It is a
------------------------------------------------------------
Medical : The patient is diagnosed with pneumonia.
Simple  : The patient has pneumonia.

Note: I've tried to preserve the original meaning and context of the sentence, while using simpler vocabulary and sentence structure.
------------------------------------------------------------
Medical : The patient is experiencing chest pain.
Simple  : The patient is feeling chest pain.

Note: This is a very bas

In [14]:
import time

medical_sentence = "The patient is suffering from hypertension."

prompt = f"""Simplify this medical sentence.

Medical: {medical_sentence}

Simple:"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

start = time.time()

outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

end = time.time()

print("Inference Time:", round(end - start, 2), "seconds")

Inference Time: 1.54 seconds


In [15]:
generated_tokens = outputs.shape[1]

tokens_per_second = generated_tokens / (end - start)

print("Generated Tokens:", generated_tokens)
print("Tokens/sec:", round(tokens_per_second, 2))

Generated Tokens: 58
Tokens/sec: 37.6


In [16]:
gpu_memory = torch.cuda.memory_allocated() / (1024 ** 2)

print("GPU Memory (MB):", round(gpu_memory, 2))

GPU Memory (MB): 2366.26


In [17]:
import psutil

ram = psutil.virtual_memory().used / (1024 ** 2)

print("RAM Usage (MB):", round(ram, 2))

RAM Usage (MB): 5090.93


In [18]:
import psutil

cpu = psutil.cpu_percent(interval=1)

print("CPU Usage (%):", cpu)

CPU Usage (%): 90.5


In [19]:
import textstat

simple_text = "The patient has high blood pressure."

readability = textstat.flesch_reading_ease(simple_text)
grade = textstat.flesch_kincaid_grade(simple_text)

print("Readability:", round(readability, 2))
print("Grade:", round(grade, 2))

Readability: 87.95
Grade: 2.48


In [20]:
from sentence_transformers import SentenceTransformer, util

model_sim = SentenceTransformer("all-MiniLM-L6-v2")

medical = "The patient is suffering from hypertension."
simple = "The patient has high blood pressure."

emb1 = model_sim.encode(medical, convert_to_tensor=True)
emb2 = model_sim.encode(simple, convert_to_tensor=True)

similarity = util.cos_sim(emb1, emb2)

print("Similarity:", round(similarity.item(), 2))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity: 0.85
